# E-Commerce Recommendation System

End-to-end demo using **Ray Data → Ray Train → Ray Data → Ray Serve** on Anyscale.

```
┌──────────────────────────────────────────────────────────────────────┐
│  Product Recommender                                                 │
│                                                                      │
│  ▼  Stage 1 — Ray Data                                               │
│       Preprocess images (resize, normalise)                          │
│       Clean text (descriptions + category)                           │
│  │                                                                   │
│  ▼  Stage 2 — Ray Train (TorchTrainer)                               │
│       Fine-tune all-MiniLM-L6-v2 with contrastive loss               │
│       Save fine-tuned model checkpoint                               │
│  │                                                                   │
│  ▼  Stage 3 — Ray Data (Batch Embedding)                             │
│       Embed entire product catalog with fine-tuned model             │
│       Save embeddings + metadata for serving                         │
│  │                                                                   │
│  ▼  Stage 4 — Ray Serve                                              │
│       POST /recommend  {image_base64}                                │
│       ImageToText  (BLIP-base)  →  caption                           │
│       ProductRecommender  (MiniLM)  →  top-5 products                │
│  │                                                                   │
│  ▼  Streamlit UI                                                     │
│       Upload image                                                   │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

**Models used**
| Stage | Model | Size | Device |
|-------|-------|------|--------|
| Train | `sentence-transformers/all-MiniLM-L6-v2` | 22 M params | CPU |
| Embed | fine-tuned MiniLM | 22 M params | CPU |
| Serve | `Salesforce/blip-image-captioning-base` | 224 M params | CPU |
| Serve | fine-tuned MiniLM | 22 M params | CPU |

Everything runs on **CPU** and completes in a few minutes.

## 0 — Setup

In [ ]:
# Install dependencies (skip if already installed)
!pip install -q -r setup/requirements.txt

In [ ]:
import json
import os
import sys
import numpy as np
from pathlib import Path
from PIL import Image

# Make sure the project root is on the path
sys.path.insert(0, str(Path().resolve()))

from utils import PRODUCTS, CATEGORIES, get_product_image

print(f"Products in catalog : {len(PRODUCTS)}")
print(f"Categories          : {CATEGORIES}")

### Peek at the product images

In [ ]:
from utils.viz import plot_category_samples

plot_category_samples(PRODUCTS, CATEGORIES, get_product_image)

---
## Stage 1 — Catalog Generation

Here we expand the base 30-product catalog to 1,000 products and add `text_clean` (strip / lowercase / collapse whitespace on `training_text`) directly in memory — no Parquet intermediate needed.

In [ ]:
# Stage 1 — build the product catalog we will embed.
#
# We start from a curated set of ~30 real product blurbs (PRODUCTS) and
# `expand_catalog` stretches that to 1,000 realistic variants — enough volume
# to exercise Ray Data without needing real product photos.
#
# `generate_catalog` materialises the records (id, name, category, description,
# image bytes, training_text). `attach_clean_text` then adds the lowercased,
# whitespace-normalised `text_clean` field the embedding model will see.
import ray
from utils import (
    PRODUCTS,
    attach_clean_text,
    expand_catalog,
    generate_catalog,
    init_ray,
)

init_ray()

raw_dir = os.path.abspath("data/raw")

products = expand_catalog(PRODUCTS, target_size=1000)
print(f"Catalog size : {len(products)} products")
print(f"Categories   : {sorted(set(p['category'] for p in products))}")

catalog_records = attach_clean_text(generate_catalog(products=products, output_dir=raw_dir))
print(f"Catalog records : {len(catalog_records)}")


---
## Stage 2 — Ray Train: Embedding Fine-Tuning

The starter model (`all-MiniLM-L6-v2`) learned from general web text, not from our product categories. That makes it easy for a fuzzy query to land near the wrong kind of item in the embedding space.

**Fine-tuning** trains the model with **contrastive loss**: same-category products move closer, different-category products move apart, so search and recommendations follow category boundaries better.

We train on **100 products (20 per category)** so each category is represented and runs stay short. The t-SNE plots below show embeddings before vs after fine-tuning.

In [ ]:
# Centralise every path the rest of the notebook writes to or reads from.
#
# `resolve_artifact_paths` prefers /mnt/cluster_storage when running on an
# Anyscale cluster (so all worker nodes see the same artefacts) and falls
# back to a local ./models folder on a laptop.
from utils import resolve_artifact_paths

paths = resolve_artifact_paths()
MODEL_OUTPUT_DIR = paths["model_dir"]
EMBEDDINGS_PATH  = paths["embeddings_path"]
METADATA_PATH    = paths["metadata_path"]
TRAIN_RESULT_DIR = paths["train_result_dir"]

print(f"MODEL_OUTPUT_DIR : {MODEL_OUTPUT_DIR}")
print(f"EMBEDDINGS_PATH  : {EMBEDDINGS_PATH}")
print(f"METADATA_PATH    : {METADATA_PATH}")
print(f"TRAIN_RESULT_DIR : {TRAIN_RESULT_DIR}")


In [ ]:
# Build a *class-balanced* training sample.
#
# Contrastive loss works by pulling same-category pairs together and pushing
# cross-category pairs apart, so every category needs enough positives. Taking
# 20 products per category gives the trainer a balanced diet and keeps one CPU
# epoch under a minute.
from utils import sample_per_category

TRAIN_PER_CATEGORY = 20
train_records = sample_per_category(
    catalog_records, n_per_category=TRAIN_PER_CATEGORY, seed=42
)
n_categories = len({r["category"] for r in train_records})
print(
    f"Sampled {len(train_records)} training records  "
    f"({n_categories} categories × ~{TRAIN_PER_CATEGORY} each)"
)


In [ ]:
# TorchTrainer: 1 worker, contrastive loss pulls same-category pairs together
import torch
from ray.train import CheckpointConfig, FailureConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from utils.training import train_loop_per_worker

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "base_model": "sentence-transformers/all-MiniLM-L6-v2",
        "epochs": 2,
        "batch_size": 8,
        "lr": 2e-5,
        "seed": 42,
        "records": train_records,
    },
    scaling_config=ScalingConfig(num_workers=1, use_gpu=torch.cuda.is_available()),
    run_config=RunConfig(
        name="ecomm_embedding_finetune",
        storage_path=os.path.abspath(TRAIN_RESULT_DIR),
        checkpoint_config=CheckpointConfig(
            num_to_keep=2,
            checkpoint_score_attribute="train_loss",
            checkpoint_score_order="min",
        ),
        failure_config=FailureConfig(max_failures=1),
    ),
)
print("TorchTrainer configured")

In [ ]:
# Run the fine-tune and persist the best checkpoint in a ready-to-serve layout.
#
# `save_best_sentence_transformer` pulls the lowest-loss checkpoint out of the
# Ray Train `Result` and writes it as a SentenceTransformer directory, which
# the serving layer can load with a single `SentenceTransformer(path)` call.
from utils.training import save_best_sentence_transformer

result = trainer.fit()
print(result.metrics_dataframe[["epoch", "train_loss"]].to_string(index=False))

save_best_sentence_transformer(result, MODEL_OUTPUT_DIR)
print(f"Model saved → {MODEL_OUTPUT_DIR}")


In [ ]:
from utils.viz import plot_training_loss

plot_training_loss(result.metrics_dataframe)

---
## Stage 3 — Ray Data: Batch Embedding

**The problem this solves:** embedding 1,000 products serially with a model loaded once per process is slow. Worse, at 1M products you can't fit the whole catalog in memory on one machine.

Ray Data solves both: each actor loads the fine-tuned model **once**, then processes a steady stream of batches. Workers run in parallel, and Ray's streaming execution means only a slice of the catalog is in memory at any time. The same code scales from 1,000 to 100M products by adjusting the `ActorPoolStrategy` size.

In [ ]:
import ray
from utils import init_ray

init_ray()

ds = ray.data.from_items(catalog_records).select_columns(
    ["product_id", "name", "category", "text_clean"]
)
print(f"Rows to embed: {ds.count()}")

In [ ]:
# Actor pool: each worker loads the fine-tuned model once and encodes batches in parallel
from pathlib import Path
from utils.embedding import ProductEmbedder

assert Path(MODEL_OUTPUT_DIR).exists() and any(Path(MODEL_OUTPUT_DIR).iterdir()), \
    f"Model not found at {MODEL_OUTPUT_DIR} — run Stage 2 first"

rows = ds.map_batches(
    ProductEmbedder,
    fn_constructor_kwargs={"model_dir": MODEL_OUTPUT_DIR},
    batch_size=8,
    num_cpus=1,
    compute=ray.data.ActorPoolStrategy(size=2),
    batch_format="numpy",
).take_all()
print(f"Embedded {len(rows)} products")

In [ ]:
# Persist vectors and metadata as two separate files.
#
# The dense (N, 384) matrix goes to .npy for fast mmap-style loads at serve
# time; product name/category/id go to a small JSON sidecar so the serving
# layer never has to pull full product text into memory at query time.
from utils.embedding import save_embeddings_and_metadata

embeddings, metadata = save_embeddings_and_metadata(rows, EMBEDDINGS_PATH, METADATA_PATH)
print(f"Embeddings : {embeddings.shape}  →  {EMBEDDINGS_PATH}")
print(f"Metadata   : {len(metadata)} products  →  {METADATA_PATH}")


In [ ]:
from utils.viz import plot_similarity_heatmap

embeddings = np.load(EMBEDDINGS_PATH)
with open(METADATA_PATH) as f:
    metadata = json.load(f)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Products indexed : {len(metadata)}")

plot_similarity_heatmap(embeddings, [m["name"] for m in metadata])

In [ ]:
import importlib

import utils.viz as _viz

importlib.reload(_viz)
from utils.viz import plot_tsne_comparison

base_embs, ft_embs, aligned_meta = plot_tsne_comparison(
    metadata, embeddings, catalog_records=catalog_records
)

In [ ]:
# Numeric check: did fine-tuning actually make the geometry category-aware?
#
# Simple "top-k neighbors share the same category" scores read ~100% on this
# demo because product names already sound like their category — so the base
# model looks strong before we even train. The two metrics below still move
# after fine-tuning and tell us the embedding space really did reshape.
from utils.viz import print_embedding_quality_report

print_embedding_quality_report(base_embs, ft_embs, aligned_meta)


---
## Stage 4 — Ray Serve: Online Recommendation API

**The problem this solves:** BLIP (224M params) and MiniLM (22M params) have very different latency, throughput, and resource profiles. Bundling them into one process means the slower model throttles the faster one.

Ray Serve deploys them as **independent deployments** that auto-scale separately. `ImageToText` can scale to 4 replicas for a spike in image uploads without adding MiniLM replicas you don't need. Both can be upgraded or rolled back independently, and the composing `RecommendationService` wires them together asynchronously behind a single `/recommend` endpoint.

In [ ]:
import os
import ray
from ray import serve
from utils import init_ray

init_ray()

# Force-assign paths so stale env vars from previous runs don't leak to workers.
os.environ["EMBEDDING_MODEL_DIR"] = MODEL_OUTPUT_DIR
os.environ["EMBEDDINGS_PATH"] = EMBEDDINGS_PATH
os.environ["METADATA_PATH"] = METADATA_PATH

import importlib, serve_app
importlib.reload(serve_app)
from serve_app import app

serve.run(app)
print("Service running at http://localhost:8000")

In [ ]:
# Smoke-test the running /recommend endpoint with a known product image.
#
# `encode_image_base64` handles JPEG compression + base64 in one call, and
# `post_recommend` POSTs the payload shape the service expects.
from utils import PRODUCTS, get_product_image, post_recommend

product = PRODUCTS[0]  # Wireless Headphones — should match other Electronics
img_arr = get_product_image(product)
result  = post_recommend(img_arr)

print(f"Caption : {result['caption']!r}")
print(f"\nTop-{len(result['recommendations'])} recommendations:")
for r in result["recommendations"]:
    print(
        f"  {r['rank']}. [{r['category']:18s}] {r['name']:35s}  sim={r['similarity']:.3f}"
    )


In [ ]:
# One image per category — does the pipeline generalise beyond headphones?
from utils import post_recommend

print(f"{'Category':<16} {'Caption':<45} {'Top recommendation'}")
print("-" * 85)
for cat in CATEGORIES:
    prod = next(p for p in PRODUCTS if p["category"] == cat)
    r = post_recommend(get_product_image(prod))
    top = r["recommendations"][0]
    print(
        f"{cat:<16} {r['caption'][:44]:<45} {top['name'][:20]} ({top['similarity']:.2f})"
    )


In [ ]:
# Show the query image alongside recommendations
from utils.viz import plot_recommendations

plot_recommendations(
    img_arr, result["recommendations"], PRODUCTS, result["caption"], get_product_image
)

In [ ]:
# Teardown (optional — keep the service running if you want to use the Streamlit app)
# serve.shutdown()

## Streamlit UI

With the service running, open a terminal and run:

```bash
streamlit run streamlit_app.py --server.port 8501 --server.address 0.0.0.0
```

Then in the workspace > Overview > Open Ports > Open (where it says Port 8501 - Streamlit) > Authenticate